In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5001/'
idp_name_2 = 'FakeCAS'
idp_username_2 = None
idp_password_2 = None
weko_user_email = None
weko_user_password = None
weko_index_name = 'GRDM-Collaboration-Test-VR-1'
default_result_path = None
close_on_fail = False
transition_timeout = 30000
browser_type = 'chromium'
ignore_https_errors = False
project_name = 'TEST-WEKO-202512052127'
project_url = None
joint_metadata_name = None
joint_metadata_name_en = 'ProjectJoint'
default_storage_label = 'NII Storage'

In [ ]:
if idp_username_2 is None:
    prompt = f"Username for {idp_name_2 or 'user account'}: "
    idp_username_2 = input(prompt)
if idp_password_2 is None:
    prompt = f"Password for {idp_username_2}: "
    idp_password_2 = getpass(prompt)

if weko_user_email is None:
    weko_user_email = input('WEKO user email: ')
if weko_user_password is None:
    weko_user_password = getpass('WEKO user password: ')

if project_name is None:
    project_name = datetime.now().strftime('TEST-WEKO-%Y%m%d%H%M')

In [ ]:
if joint_metadata_name is None:
    joint_metadata_name = datetime.now().strftime('プロジェクト連携-%Y%m%d%H%M')


# 書誌情報・根拠データをまとめてWEKOへ送信する

- サブシステム名: アドオン
- ページ/アドオン: Metadata, WEKO
- 機能分類: メタデータの送信
- シナリオ名: 書誌情報と根拠データを関連付けてエクスポート
- 用意するテストデータ: 書誌情報Notebookと根拠データNotebookの実行結果（論文原稿・根拠データのフォルダとファイルメタデータが登録済みであること）
- 事前条件: 既存ユーザー1のアカウント、WEKO教員以上のアカウント、JAIRO Cloud のインデックス


In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

In [ ]:
import yaml
import json
import importlib
import os

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm
from scripts.metadata_v2025 import ProjectMetadataForm, FileMetadataForm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path, ignore_https_errors=ignore_https_errors)

## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count():
        await consent_button.click()
    await expect(page.locator('//button[text() = "ログイン"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step, new_page=True)

## IdPを利用し、既存ユーザー1としてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_2, idp_username_2, idp_password_2, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

作成したプロジェクトのプロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    project_link = page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]')
    await expect(project_link).to_be_visible(timeout=transition_timeout)
    await project_link.click()
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    global project_url
    project_url = page.url

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「メタデータ」をクリックする

メタデータの一覧ページが表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[contains(text(), "メタデータ")]').click()
    await expect(page.locator('//button[@data-test-new-metadata-button]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「新規メタデータ」をクリックする

スキーマ選択ダイアログが表示されること


In [ ]:
import asyncio

async def _step(page):
    await page.locator('//button[@data-test-new-metadata-button]').click()
    await expect(page.locator('//*[@data-test-new-report-modal-schema="公的資金による研究データのメタデータ登録"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//button[text()="メタデータを作成"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(3)

await run_pw(_step)


## 「メタデータを作成」をクリックする

メタデータ編集ウィンドウが表示されること


In [ ]:
async def _step(page):
    await page.locator('//button[text() = "メタデータを作成"]').click()
    await expect(page.locator('//*[contains(text(), "資金配分機関情報")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 必須入力欄に正常系の値を入力する

正常系の値は以下の通り:
- 資金配分機関情報: 国立研究開発法人日本医療研究開発機構 | AMED
- 体系的番号: TEST123456
- プロジェクト名 (日本語): `joint_metadata_name`
- Project name (English): `joint_metadata_name_en`
- プロジェクトの分野: 社会基盤 | 789

各フィールドに入力した値が表示されること


In [ ]:
async def _step(page):
    form = ProjectMetadataForm(page)
    await form.fill("資金配分機関情報", "国立研究開発法人日本医療研究開発機構 | AMED")
    await form.fill_power_select_by_search("体系的番号", "TEST123456")
    await form.fill("プロジェクト名 (日本語)", joint_metadata_name)
    await form.fill("Project name (English)", joint_metadata_name_en)
    await form.fill("プロジェクトの分野", "社会基盤 | 789")
    assert await form.get("プロジェクト名 (日本語)") == joint_metadata_name

await run_pw(_step)


## 「次へ」をクリックする

左側ペイン「メタデータ登録」が緑色かつチェックマークが表示されること


In [ ]:
import asyncio

async def _step(page):
    await page.locator('//a[@data-test-goto-next-page]').click()
    await expect(page.locator('//a[contains(@data-test-link, "メタデータ登録")]//i[contains(@class, "fa-check")]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(10)

await run_pw(_step)


## プロジェクトダッシュボードに戻る

プロジェクトのファイル一覧が表示されること


In [ ]:
async def _step(page):
    await page.goto(project_url)
    await expect(grdm.get_select_storage_title_locator(page, default_storage_label)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「論文原稿」をクリックし、「メタデータ登録」をクリックする

「メタデータ登録」ダイアログが表示されること


In [ ]:
async def _step(page):
    await grdm.get_select_storage_title_locator(page, default_storage_label).click()
    target_label = f'JAIRO Cloud: {weko_index_name}'
    await grdm.get_select_storage_title_locator(page, target_label).click()
    folder_locator = grdm.get_select_folder_title_locator(page, '論文原稿')
    await expect(folder_locator).to_be_visible(timeout=transition_timeout)
    await folder_locator.click()
    await page.locator('//*[text() = "メタデータ登録"]').click()
    await expect(page.locator('//input[starts-with(@id, "draft-")]')).to_have_count(5, timeout=transition_timeout)

await run_pw(_step)


## {joint_metadata_name} をチェックし、「選択」をクリックする

「開く」ボタンが表示されること

In [ ]:
async def _step(page):
    target = page.locator(f'//*[text() = "{joint_metadata_name}"]')
    checkbox = target.locator('..//input[@type = "checkbox"]')
    await checkbox.click()
    await expect(checkbox).to_be_checked(timeout=transition_timeout)
    await page.locator('//*[text() = "選択"]').click()
    await expect(page.locator('//*[text() = "開く"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(3)

await run_pw(_step)


## 「閉じる」をクリックする

In [ ]:
async def _step(page):
    close_button = page.locator(f'//*[text() = "閉じる" and contains(@class, "btn-success")]')
    await close_button.click()
    await expect(close_button).not_to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(3)

await run_pw(_step)


## 「根拠データ」をクリックし、「メタデータ登録」をクリックする

「メタデータ登録」ダイアログが表示されること


In [ ]:
async def _step(page):
    await grdm.get_select_storage_title_locator(page, default_storage_label).click()
    target_label = f'JAIRO Cloud: {weko_index_name}'
    await grdm.get_select_storage_title_locator(page, target_label).click()
    folder_locator = grdm.get_select_folder_title_locator(page, '根拠データ')
    await expect(folder_locator).to_be_visible(timeout=transition_timeout)
    await folder_locator.click()
    await page.locator('//*[text() = "メタデータ登録"]').click()
    await expect(page.locator('//input[starts-with(@id, "draft-")]')).to_have_count(5, timeout=transition_timeout)
    await asyncio.sleep(3)

await run_pw(_step)


## {joint_metadata_name} をチェックし、「選択」をクリックする

「開く」ボタンが表示されること

In [ ]:
async def _step(page):
    target = page.locator(f'//*[text() = "{joint_metadata_name}"]')
    checkbox = target.locator('..//input[@type = "checkbox"]')
    await checkbox.click()
    await expect(checkbox).to_be_checked(timeout=transition_timeout)
    await page.locator('//*[text() = "選択"]').click()
    await expect(page.locator('//*[text() = "開く"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(3)

await run_pw(_step)


## 「開く」をクリックして、「次へ」をクリックする

登録データ一覧に「論文原稿」と「根拠データ」が表示されること


In [ ]:
async def _step(page):
    await page.locator('//*[text() = "開く"]').click()
    await expect(page.locator('//*[@data-test-goto-review]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//p[contains(., "論文原稿")]/../..//*[contains(@class, "fa-check-square")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//p[contains(., "根拠データ")]/../..//*[contains(@class, "fa-check-square")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「内容確認」をクリックする

左側ペイン「登録データ」が緑色かつチェックマークが表示されること


In [ ]:
import re

async def _step(page):
    await page.locator('//*[@data-test-goto-review]').click()
    await expect(page.locator('//span[@data-test-label and text() = "登録データ"]/../preceding-sibling::i')).to_have_class(re.compile(r'.*fa-check-circle-o.*'), timeout=transition_timeout)

await run_pw(_step)


## 「エクスポート」をクリックする

エクスポート先選択画面が表示されること


In [ ]:
async def _step(page):
    await page.locator('//button[@data-test-registration-card-export]').click()
    await expect(page.locator('//select[@id = "registration-report-format-selection"]')).to_be_enabled(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)


## 「リポジトリ (JAIRO Cloud)に登録 - {weko_index_name}」を選択する

WEKO送信先のオプションが表示されること


In [ ]:
async def _step(page):
    target_option = f'リポジトリ (JAIRO Cloud)に登録 - {weko_index_name}'
    select = page.locator('//*[@id = "registration-report-format-selection"]')
    await select.select_option(label=target_option)

await run_pw(_step)


## 「エクスポート」をクリックしてWEKO送信を開始する

WEKO送信ボタンを押して処理が始まること


In [ ]:
async def _step(page):
    submit_button = page.locator('//*[@data-test-registration-report-submit]')
    await expect(submit_button).to_be_enabled(timeout=transition_timeout)
    popup_future = page.context.wait_for_event('page', timeout=transition_timeout * 5)
    await submit_button.click()
    await expect(submit_button).to_contain_text('エクスポート中', timeout=transition_timeout * 5)
    popup = await popup_future
    await popup.wait_for_load_state()
    await expect(popup.locator('//input[@name = "email"]')).to_be_visible(timeout=transition_timeout)
    return popup

await run_pw(_step)


## WEKOにログインし、アイテムの関連情報を確認する

論文原稿に含まれるファイルの名称とsample dataへのリンクが表示されること


In [ ]:
weko_manuscript_url = None

async def _step(page):
    await page.locator('//input[@name = "email"]').fill(weko_user_email)
    await page.locator('//input[@name = "password"]').fill(weko_user_password)
    await page.locator('//button[@type = "submit"]').click()
    await expect(page.locator('//a[contains(text(), "JSON")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[contains(text(), "paper_draft.docx")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[contains(text(), "paper_supplement.pdf")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//a[text() = "sample data"]')).to_be_visible(timeout=transition_timeout)
    global weko_manuscript_url
    weko_manuscript_url = page.url

await run_pw(_step)


## 「sample data」へのリンクをクリックする

根拠データに含まれるファイルの名称が表示されること

In [ ]:
weko_data_url = None

async def _step(page):
    await page.locator('//a[text() = "sample data"]').click()
    await expect(page.locator('//a[contains(text(), "JSON")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[contains(text(), "evidence_notes.docx")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[contains(text(), "evidence_overview.txt")]')).to_be_visible(timeout=transition_timeout)
    global weko_data_url
    weko_data_url = page.url

await run_pw(_step)


## 「Delete」をクリックする

アイテムが削除されること


In [ ]:
async def _step(page):
    await page.locator('//a[@id = "btn_delete"]').click()
    await expect(page.locator('//button[@id = "confirm_delete_button"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//button[@id = "confirm_delete_button"]').click()
    await expect(page.locator('//*[contains(text(), "This item has been deleted")]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(transition_timeout / 1000 / 2)

await run_pw(_step)


## 書誌情報アイテムのページに戻り、「Delete」をクリックする

In [ ]:
async def _step(page):
    await page.goto(weko_manuscript_url)
    await expect(page.locator('//a[contains(text(), "JSON")]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//a[@id = "btn_delete"]').click()
    await expect(page.locator('//button[@id = "confirm_delete_button"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//button[@id = "confirm_delete_button"]').click()
    await expect(page.locator('//*[contains(text(), "This item has been deleted")]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(transition_timeout / 1000 / 2)

await run_pw(_step)


In [ ]:
await close_latest_page()

後始末


In [ ]:
await finish_pw_context(screenshot=False, last_path=default_result_path)

In [ ]:
!rm -fr {work_dir}